# Suite VIS — Visitors protocol conformance

Every class that plays a visitor role must implement its `Visitors` protocol completely, with matching arity:
Proxies builders, Plain writers, the Reachable collector, and the Validators recorder. Proxies instances implement
`Visitable`, not a visitor protocol.

In [ ]:
import typing

from schemas.Framework import Plain, Proxies, Reachable, Validators, Visitors
from support import conformance_problems, instance_problems, protocol_methods

## VIS-01 · The protocols are `typing.Protocol`s and declare what Framework.md says

In [ ]:
for name in ["OfAny", "OfNative", "OfProperty", "OfObject", "OfAdjacency", "OfEntry", "OfLink", "OfRelation",
             "OfUnion", "OfIntersection", "Visitable"]:
    protocol = getattr(Visitors, name)
    assert typing.Protocol in protocol.__mro__, name
assert set(protocol_methods(Visitors.OfAny)) == {"as_native", "as_object", "as_union", "as_intersection"}
assert set(protocol_methods(Visitors.Visitable)) == {"identity", "schema_name", "accept"}
assert set(protocol_methods(Visitors.OfObject)) == {"properties", "has", "property", "clear", "adjacencies", "adjacency"}
assert "identity" not in protocol_methods(Visitors.OfObject)  # identity belongs to the visited object
assert set(protocol_methods(Visitors.OfLink)) == {"name", "target", "set"}
assert Visitors.Native == int | float | str | bool | bytes

## VIS-02 · Every implementation conforms to its protocol

In [ ]:
V = Visitors
implementations = {
    "Proxies": [(Proxies._ObjectBuilder, V.OfObject), (Proxies._AdjacencySlot, V.OfAdjacency),
                (Proxies._EntryBuilder, V.OfEntry), (Proxies._LinkSlot, V.OfLink),
                (Proxies._PropertySlot, V.OfProperty), (Proxies._AnySlot, V.OfAny), (Proxies._NativeSlot, V.OfNative),
                (Proxies._ObjectData, V.Visitable)],
    "Plain": [(Plain._ObjectWriter, V.OfObject), (Plain._AdjacencyWriter, V.OfAdjacency),
              (Plain._EntryWriter, V.OfEntry), (Plain._LinkWriter, V.OfLink), (Plain._PropertyWriter, V.OfProperty),
              (Plain._AnyWriter, V.OfAny), (Plain._NativeWriter, V.OfNative)],
    "Reachable": [(Reachable._Collector, V.OfObject), (Reachable._Adjacency, V.OfAdjacency),
                  (Reachable._Entry, V.OfEntry), (Reachable._Link, V.OfLink), (Reachable._Ignored, V.OfProperty),
                  (Reachable._Ignored, V.OfAny), (Reachable._Ignored, V.OfNative)],
    "Validators": [(Validators._ObjectRecord, V.OfObject), (Validators._AdjacencyRecord, V.OfAdjacency),
                   (Validators._EntryRecord, V.OfEntry), (Validators._Link, V.OfLink),
                   (Validators._Value, V.OfProperty), (Validators._Value, V.OfAny), (Validators._Value, V.OfNative)],
}
problems = [p for pairs in implementations.values() for impl, proto in pairs for p in conformance_problems(impl, proto)]
assert problems == [], problems

## VIS-03 · Proxies instances are `Visitable`, not visitors
Framework.md: builders and serializers implement `Visitors`; proxies do not.

In [ ]:
for protocol in [V.OfObject, V.OfAdjacency, V.OfEntry]:
    assert conformance_problems(Proxies._ObjectData, protocol), protocol

## VIS-04 · Visitor methods chain: non-query methods return `self`

In [ ]:
from schemas.Framework import Schemas as S
from support import text

Rel = S.OfRelation.Builder().links("a", "b").properties(text("k")).create()
Obj = S.OfObject.Builder().properties(text("x")).relations(lambda r: r.name("r").of(Rel).me("a")).create()
Proxies.register("Obj", Obj)
Proxies.register("Rel", Rel)

builder = Proxies.Builders.Obj()
assert builder.property("x", lambda p: None) is builder
assert builder.properties(lambda p: None) is builder
assert builder.clear("x") is builder
assert builder.adjacencies(lambda a: None) is builder
assert builder.adjacency("r", lambda a: None) is builder

slots = []
builder.property("x", slots.append)
prop = slots[0]
assert prop.value(lambda a: None) is prop and prop.clear() is prop
prop.value(slots.append)
any_ = slots[1]
assert any_.as_native(lambda n: None) is any_
any_.as_native(slots.append)
native = slots[2]
assert native.set("v") is native and native.clear() is native

builder.adjacency("r", slots.append)
adjacency = slots[3]
assert adjacency.add(lambda e: slots.append(e)) is adjacency
entry = slots[4]
assert entry.property("k", lambda p: None) is entry and entry.clear("k") is entry
assert entry.links(lambda link: slots.append(link)) is entry
link = slots[5]
assert link.name() == "b" and link.set(Proxies.Builders.Obj().create()) is link
assert adjacency.remove(entry) is adjacency and adjacency.entries(lambda e: None) is adjacency

## VIS-05 · Protocols without implementations yet
`OfUnion`, `OfIntersection` and `OfRelation` are declared but no class implements them; union, intersection and
object-valued properties raise `NotImplementedError` in every implementation (pinned here, see TestPlan.md).

In [ ]:
from support import raises

for impl in [Proxies._AnySlot, Plain._AnyWriter, Validators._Value]:
    instance = impl.__new__(impl)
    for method in ["as_object", "as_union", "as_intersection"]:
        with raises(NotImplementedError):
            getattr(impl, method)(instance, lambda v: None)

## VIS-06 · Live instances conform too
A class can declare every method yet have an instance attribute shadow one of them: the recorder in `Validators`
once did (its `adjacencies` dict hid its `adjacencies()` method), and VIS-02, which inspects classes, missed it.

In [ ]:
found = []
builder = Proxies.Builders.Obj()
found.append((builder, V.OfObject))


def on_property(p):
    found.append((p, V.OfProperty))
    p.value(lambda a: (found.append((a, V.OfAny)), a.as_native(lambda n: found.append((n, V.OfNative)))))


def on_adjacency(a):
    found.append((a, V.OfAdjacency))
    a.add(lambda e: (found.append((e, V.OfEntry)), e.link("b", lambda k: found.append((k, V.OfLink)))))


builder.property("x", on_property)
builder.adjacency("r", on_adjacency)
found.append((Proxies.Builders.Obj().create(), V.Visitable))
record = Validators._ObjectRecord()
record.adjacency("r", lambda a: a.add(lambda e: found.append((e, V.OfEntry))))
found += [(record, V.OfObject), (Reachable._Collector(lambda target: None), V.OfObject)]
writer = Plain._ObjectWriter({}, Obj, lambda target: {})
found.append((writer, V.OfObject))
writer.adjacency("r", lambda a: (found.append((a, V.OfAdjacency)), a.add(lambda e: found.append((e, V.OfEntry)))))
problems = [p for instance, protocol in found for p in instance_problems(instance, protocol)]
assert problems == [], problems
assert record.adjacencies(lambda a: a.name()) is record  # the method the bug hid